# AEGIS-Pipe — Cell 1: immutable configuration and raw-data audit

This cell performs only lightweight reads. It never loads a complete WFS file. The independent experimental unit is the **physical pipe** (B–E), so all channels from a held-out pipe must remain together in every later split.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import platform
import re
import shutil
import sys

import pandas as pd
from IPython.display import display

# -----------------------------------------------------------------------------
# Project paths — edit only RAW if the dataset is moved.
# -----------------------------------------------------------------------------
RAW = Path(r"D:\Pipeline RUL Data\data\raw")
PROJECT = RAW.parents[1]                 # D:\Pipeline RUL Data
INTERIM = PROJECT / "data" / "interim"
PROCESSED = PROJECT / "data" / "processed"
CACHE = PROJECT / "cache"
RUNS = PROJECT / "runs"
FIGURES = PROJECT / "figures_main"

for directory in (INTERIM, PROCESSED, CACHE, RUNS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class ExperimentConfig:
    pipe_ids: tuple[str, ...] = ("B", "C", "D", "E")
    active_ae_channels: tuple[int, ...] = (5, 6, 7, 8)
    nominal_sample_rate_hz: int = 1_000_000
    sample_dtype: str = "<i2"             # hypothesis; verify before decoding
    record_length_dtype: str = "<u2"      # verified from the supplied header
    stream_record_id: int = 0xAE
    header_probe_bytes: int = 4096
    split_group: str = "pipe_id"           # never split by channel/window
    random_seed: int = 20260907


CFG = ExperimentConfig()

if not RAW.is_dir():
    raise FileNotFoundError(f"Raw-data directory does not exist: {RAW}")

found = {path.stem.upper(): path for path in RAW.glob("*.wfs")}
missing = [pipe_id for pipe_id in CFG.pipe_ids if pipe_id not in found]
if missing:
    raise FileNotFoundError(f"Missing WFS files for pipes: {missing}")

files = [found[pipe_id] for pipe_id in CFG.pipe_ids]
extra = sorted(set(found).difference(CFG.pipe_ids))
if extra:
    print(f"Note: ignoring additional WFS stems: {extra}")

rows = []
for path in files:
    with path.open("rb") as stream:
        header = stream.read(CFG.header_probe_bytes)

    version_match = re.search(rb"Version\s+V([0-9.]+)", header)
    version = version_match.group(1).decode("ascii") if version_match else None
    rows.append(
        {
            "pipe_id": path.stem.upper(),
            "file": path.name,
            "size_bytes": path.stat().st_size,
            "size_GiB": path.stat().st_size / 2**30,
            "express8_signature": b"Express-8" in header,
            "software_version": version,
            "header_sha256_4KiB": hashlib.sha256(header).hexdigest(),
        }
    )

manifest = pd.DataFrame(rows).set_index("pipe_id")
manifest_path = RUNS / "raw_wfs_manifest.csv"
config_path = RUNS / "experiment_config.json"
manifest.to_csv(manifest_path)
config_path.write_text(
    json.dumps(
        {
            **asdict(CFG),
            "raw_directory": str(RAW),
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "python": sys.version.split()[0],
            "platform": platform.platform(),
        },
        indent=2,
    ),
    encoding="utf-8",
)

disk = shutil.disk_usage(PROJECT)
total_gib = manifest["size_GiB"].sum()
free_gib = disk.free / 2**30

display(manifest[["file", "size_GiB", "express8_signature", "software_version"]].round({"size_GiB": 2}))
print(f"\nRaw total: {total_gib:,.2f} GiB")
print(f"Free space on {PROJECT.drive or PROJECT.anchor}: {free_gib:,.2f} GiB")
print(f"Manifest: {manifest_path}")
print(f"Config:   {config_path}")
print("\nSplit invariant: pipe_id is the independent unit; channels 5–8 stay together.")

assert manifest["express8_signature"].all(), "At least one file lacks the Express-8 signature."
assert manifest["software_version"].notna().all(), "Could not recover the WFS software version."
assert manifest.index.is_unique, "Duplicate physical pipe IDs detected."
